# M1.3 Electric Potential and the Gradient

Book pointer: chapter 3, sections 3.1 (electric potential), 3.2 (properties), 3.3 (potentials from charge distributions), 3.4 (relation to the field), 3.5 (applications).

## 1. Recall

1. Why does a conservative force allow a potential energy? Which property of the electrostatic field makes it conservative?
2. Write $V(\mathbf{r}_b) - V(\mathbf{r}_a)$ as a line integral of $\mathbf{E}$. What is the sign and why?
3. Write $V(\mathbf{r})$ for a single point charge and for a volume charge density, with $V = 0$ at infinity.
4. State $\mathbf{E} = -\nabla V$. What is the geometric relation between equipotential surfaces and field lines?
5. Numerically, why is it cheaper to compute $V$ on a grid and then differentiate than to compute $\mathbf{E}$ directly?

## 2. Principle to algorithm

The potential of point charges is a scalar sum with one power of distance less than the field:

$$V(\mathbf{r}) = \frac{1}{4\pi\epsilon_0}\sum_i \frac{q_i}{|\mathbf{r} - \mathbf{r}_i|}.$$

No vector components, no cross terms. That makes it a third of the work of the field sum and much easier to read off a contour plot. The field comes back by differentiating: on a grid, $\mathbf{E} = -\nabla V$ with the same central differences as chapter 1.

Two checks are available at every step. First, the numerical $-\nabla V$ must match the direct superposition field from chapter 2 away from the charges. Second, the line integral of $\mathbf{E}$ between two points must equal the potential difference regardless of path.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import grids, fields

plt.rcParams["figure.figsize"] = (6, 5)
K = fields.K

In [ ]:
# Same dipole as chapter 2.
q = 1e-9
src = np.array([[0, 0.5, 0], [0, -0.5, 0]])
chg = np.array([q, -q])

x, y, P = grids.grid2d((-2, 2), (-2, 2), 161)
hx, hy = grids.spacing(x, y)
X, Y, _ = grids.components(P)

V = fields.point_charges_V(src, chg, P)
E_from_V = -fields.gradient(V, hx, hy)        # shape (nx, ny, 2)

levels = np.linspace(-30, 30, 31)
cs = plt.contour(X, Y, V, levels=levels, cmap="coolwarm")
plt.clabel(cs, fmt="%.0f", fontsize=7)
step = 8
plt.quiver(X[::step, ::step], Y[::step, ::step], E_from_V[::step, ::step, 0], E_from_V[::step, ::step, 1],
           angles="xy", color="k", alpha=0.6)
plt.gca().set_aspect("equal"); plt.title("equipotentials of a dipole with E = -grad V"); plt.show()

In [ ]:
# Check 1: -grad V against direct superposition, away from the charges.
E_direct = fields.point_charges_E(src, chg, P)[..., :2]
dist_to_nearest = np.min([grids.magnitude(P - s) for s in src], axis=0)
mask = dist_to_nearest > 0.3

rel = grids.magnitude(E_from_V - E_direct) / grids.magnitude(E_direct)
print(f"grid step {hx:.4f}.  Relative error of -grad V versus direct E, at least 0.3 from any charge:")
print(f"  median {np.median(rel[mask]):.2e}   max {rel[mask].max():.2e}")

plt.imshow(np.where(mask, np.log10(rel), np.nan).T, origin="lower", extent=(-2, 2, -2, 2), cmap="magma")
plt.colorbar(label="log10 relative error"); plt.title("where the finite difference is accurate"); plt.show()

In [ ]:
# Check 2: path independence. Integrate E along two different paths from a to b and compare to V(a) - V(b).
a = np.array([-1.5, -1.0, 0.0])
b = np.array([1.2, 1.4, 0.0])
E_func = lambda pts: fields.point_charges_E(src, chg, pts)

t = np.linspace(0, 1, 4001)[:, None]
straight = a + t * (b - a)
corner = np.vstack([a + t * np.array([b[0] - a[0], 0, 0]), np.array([b[0], a[1], 0]) + t * np.array([0, b[1] - a[1], 0])])
bulge = a + t * (b - a) + np.sin(np.pi * t) * np.array([1.0, -1.0, 0]) * 1.2

dV = fields.point_charges_V(src, chg, a[None]) [0] - fields.point_charges_V(src, chg, b[None])[0]
for name, path in (("straight", straight), ("corner", corner), ("bulge", bulge)):
    print(f"  integral of E . dl along {name:8s}: {fields.line_integral(E_func, path):+.6f} V")
print(f"  V(a) - V(b)                         : {dV:+.6f} V")

### Exercise: potential of a ring on its axis

For a ring of radius $R$ and charge $Q$, the potential on the axis is $V(z) = \dfrac{1}{4\pi\epsilon_0}\dfrac{Q}{\sqrt{z^2 + R^2}}$.

Replace the body of `ring_V_axis` with your own scalar sum over $N$ point charges on the ring. Then differentiate the result numerically along $z$ and compare with the closed-form $E_z$ from chapter 2.

In [ ]:
def ring_V_axis(z, R, Q, N):
    """Potential on the axis of a ring built from N point charges. z is a 1D array."""
    phi = (np.arange(N) + 0.5) * 2 * np.pi / N
    pos = np.stack([R * np.cos(phi), R * np.sin(phi), np.zeros(N)], axis=-1)
    obs = np.stack([0 * z, 0 * z, z], axis=-1)
    return fields.point_charges_V(pos, np.full(N, Q / N), obs)   # replace with your own sum


R, Q = 1.0, 1e-9
z = np.linspace(-3, 3, 601)
Vz = ring_V_axis(z, R, Q, 64)
assert np.allclose(Vz, K * Q / np.sqrt(z**2 + R**2), rtol=1e-6)

Ez_from_V = -np.gradient(Vz, z[1] - z[0])
Ez_exact = K * Q * z / (z**2 + R**2) ** 1.5
plt.plot(z, Ez_exact, "k-", lw=3, alpha=0.3, label="E_z closed form")
plt.plot(z, Ez_from_V, "C2--", label="-dV/dz numerical")
plt.xlabel("z / R"); plt.legend(); plt.show()
print("ring potential check passed")

## 4. Verify

How the error of $-\nabla V$ shrinks with grid step. A central difference has error proportional to $h^2$. Refine the grid for a single point charge and measure the maximum relative error at points more than $0.5$ away from the charge. The slope on log-log axes should be close to $2$.

In [ ]:
hs, errs = [], []
for n in (21, 41, 81, 161, 321):
    xg, yg, Pg = grids.grid2d((0.5, 2.5), (0.5, 2.5), n)
    h = grids.spacing(xg, yg)
    Vg = fields.point_charges_V([[0, 0, 0]], [q], Pg)
    Eg = -fields.gradient(Vg, *h)
    Ex = fields.point_charges_E([[0, 0, 0]], [q], Pg)[..., :2]
    inner = (slice(1, -1),) * 2
    errs.append((grids.magnitude(Eg - Ex) / grids.magnitude(Ex))[inner].max())
    hs.append(h[0])
hs, errs = np.array(hs), np.array(errs)
slope = np.polyfit(np.log(hs), np.log(errs), 1)[0]
plt.loglog(hs, errs, "o-"); plt.xlabel("grid step h"); plt.ylabel("max relative error of -grad V")
plt.title(f"observed order: slope = {slope:.2f}"); plt.grid(True, which="both", alpha=0.3); plt.show()
assert 1.8 < slope < 2.2

## 5. Explore and rebuild

Explore. Build a monopole ($+q$), a dipole ($+q$, $-q$ separated by $d$) and a linear quadrupole ($+q$, $-2q$, $+q$ along a line). Sample $|V|$ along a direction that is not a symmetry axis, from $r = 5d$ to $r = 500d$, and fit the slope on log-log axes. Explain the three slopes you find. Then repeat for $|\mathbf{E}|$.

Rebuild. Close this notebook and write `point_charges_V`, the gradient-based field, and the three-path check from a blank editor.